  numeric_columns:
    - Tenure Months
    - Monthly Charges
    - Total Charges
    - Churn Score
    - CLTV
    - Latitude
    - Longitude
  categorical_columns:
    - Gender
    - Senior Citizen
    - Partner
    - Dependents
    - Phone Service
    - Multiple Lines
    - Internet Service
    - Online Security
    - Online Backup
    - Device Protection
    - Tech Support
    - Streaming TV
    - Streaming Movies
    - Contract
    - Paperless Billing
    - Payment Method
    - Churn Label

In [1]:
import os

In [2]:
%pwd

'd:\\End_to_End_ML_project_for_Customer_Churn_Prediction\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\End_to_End_ML_project_for_Customer_Churn_Prediction'

1. Load raw data
2. Drop unnecessary columns
3. Feature engineering
4. Save cleaned dataset

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataPreprocessingConfig:
    root_dir: Path
    raw_data_dir: Path
    input_file_name: str
    processed_data_file: Path
    drop_columns: list

In [6]:
from src.customer_churn_prediction.constants import *
from src.customer_churn_prediction.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH, params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_preprocessing_config(self) -> DataPreprocessingConfig:
        config = self.config.data_preprocessing

        create_directories([config.root_dir])

        data_preprocessing_config = DataPreprocessingConfig(
            root_dir=config.root_dir,
            raw_data_dir=config.raw_data_dir,
            input_file_name=config.input_file_name,
            processed_data_file=config.processed_data_file,
            drop_columns=list(config.drop_columns)
        )

        return data_preprocessing_config

In [9]:
import os
import pickle
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from src.customer_churn_prediction import logger

In [ ]:
import os
import pickle
import pandas as pd

from sklearn.preprocessing import StandardScaler
from src.customer_churn_prediction import logger

class DataPreprocessing:

    def __init__(self, config: DataPreprocessingConfig):

        self.config = config
        self.scaler = StandardScaler()

    # =========================================================
    # Load Data
    # =========================================================

    def get_raw_file_path(self) -> str:

        return os.path.join(
            self.config.raw_data_dir,
            self.config.input_file_name
        )

    def load_data(self) -> pd.DataFrame:

        raw_file_path = self.get_raw_file_path()

        if not os.path.exists(raw_file_path):

            raise FileNotFoundError(
                f"Raw data file not found: {raw_file_path}"
            )

        logger.info(f"Loading raw data from: {raw_file_path}")

        data = pd.read_csv(raw_file_path)

        logger.info(f"Dataset loaded successfully: {data.shape}")

        return data

    # =========================================================
    # Drop Columns
    # =========================================================

    def drop_columns(self, data: pd.DataFrame) -> pd.DataFrame:

        drop_cols = [
            col for col in self.config.drop_columns
            if col in data.columns
        ]

        if drop_cols:

            logger.info(f"Dropping columns: {drop_cols}")

            data = data.drop(columns=drop_cols)

        return data

    # =========================================================
    # Create Derived Features
    # =========================================================

    def create_derived_features(
        self,
        data: pd.DataFrame
    ) -> pd.DataFrame:

        logger.info("Creating derived features...")

        # Convert Total Charges
        if "Total Charges" in data.columns:

            data["Total Charges"] = pd.to_numeric(
                data["Total Charges"],
                errors="coerce"
            )

        # Average monthly spend
        if all(
            col in data.columns
            for col in ["Total Charges", "Tenure Months"]
        ):

            data["Avg Monthly Spend"] = (
                data["Total Charges"] /
                (data["Tenure Months"] + 1)
            )

            logger.info(
                "Created feature: Avg Monthly Spend"
            )

        # Long-term customer
        if "Tenure Months" in data.columns:

            data["LongTermCustomer"] = (
                data["Tenure Months"] >= 24
            ).astype(int)

            logger.info(
                "Created feature: LongTermCustomer"
            )

        # High monthly charge
        if "Monthly Charges" in data.columns:

            threshold = data["Monthly Charges"].median()

            data["HighMonthlyCharges"] = (
                data["Monthly Charges"] > threshold
            ).astype(int)

            logger.info(
                "Created feature: HighMonthlyCharges"
            )

        # Total subscribed services
        service_cols = [
            "Phone Service",
            "Online Security",
            "Online Backup",
            "Device Protection",
            "Tech Support",
            "Streaming TV",
            "Streaming Movies"
        ]

        existing_service_cols = [
            col for col in service_cols
            if col in data.columns
        ]

        if existing_service_cols:

            data["TotalServices"] = data[
                existing_service_cols
            ].apply(
                lambda row: sum(row == "Yes"),
                axis=1
            )

            logger.info(
                "Created feature: TotalServices"
            )

        logger.info(
            f"Derived feature creation completed: "
            f"{data.shape}"
        )

        return data

    # =========================================================
    # Handle Missing Values
    # =========================================================

    def impute_missing_values(
        self,
        data: pd.DataFrame
    ) -> pd.DataFrame:

        logger.info("Handling missing values...")

        # Numeric columns
        for col in self.config.numeric_columns:

            if col in data.columns:

                if data[col].isnull().sum() > 0:

                    median_value = data[col].median()

                    data[col] = data[col].fillna(
                        median_value
                    )

                    logger.info(
                        f"Filled missing values in "
                        f"{col} with median"
                    )

        # Categorical columns
        for col in self.config.categorical_columns:

            if col in data.columns:

                if data[col].isnull().sum() > 0:

                    mode_value = data[col].mode()

                    fill_value = (
                        mode_value.iloc[0]
                        if not mode_value.empty
                        else "Unknown"
                    )

                    data[col] = data[col].fillna(
                        fill_value
                    )

                    logger.info(
                        f"Filled missing values in "
                        f"{col} with mode"
                    )

        return data

    # =========================================================
    # Encode Categorical Features
    # =========================================================

    def encode_categorical_columns(
        self,
        data: pd.DataFrame
    ) -> pd.DataFrame:

        categorical_cols = [

            col for col in self.config.categorical_columns

            if col in data.columns
        ]

        if categorical_cols:

            logger.info(
                f"Encoding categorical columns: "
                f"{categorical_cols}"
            )

            data = pd.get_dummies(
                data,
                columns=categorical_cols,
                drop_first=True,
                dtype=int
            )

            logger.info(
                f"Categorical encoding completed: "
                f"{data.shape}"
            )

        return data

    # =========================================================
    # Scale Numeric Features
    # =========================================================

    def scale_numeric_columns(
        self,
        data: pd.DataFrame
    ) -> pd.DataFrame:

        numeric_cols = [

            col for col in self.config.numeric_columns

            if col in data.columns
        ]

        derived_numeric = [
            "Avg Monthly Spend",
            "TotalServices"
        ]

        numeric_cols += [

            col for col in derived_numeric

            if col in data.columns
            and col not in numeric_cols
        ]

        if numeric_cols:

            logger.info(
                f"Scaling numeric columns: "
                f"{numeric_cols}"
            )

            self.scaler.fit(data[numeric_cols])

            data[numeric_cols] = self.scaler.transform(
                data[numeric_cols]
            )

            logger.info(
                "Numeric feature scaling completed"
            )

            self._save_scaler()

        return data

    # =========================================================
    # Save Scaler
    # =========================================================

    def _save_scaler(self) -> None:

        scaler_path = os.path.join(
            self.config.root_dir,
            "scaler.pkl"
        )

        os.makedirs(
            self.config.root_dir,
            exist_ok=True
        )

        with open(scaler_path, "wb") as f:

            pickle.dump(self.scaler, f)

        logger.info(
            f"Scaler saved at: {scaler_path}"
        )

    # =========================================================
    # Save Processed Data
    # =========================================================

    def save_processed_data(
        self,
        data: pd.DataFrame
    ) -> None:

        os.makedirs(
            self.config.root_dir,
            exist_ok=True
        )

        logger.info(
            f"Saving processed data to: "
            f"{self.config.processed_data_file}"
        )

        data.to_csv(
            self.config.processed_data_file,
            index=False
        )

        logger.info(
            f"Processed data saved successfully: "
            f"{data.shape}"
        )

    # =========================================================
    # Main Pipeline
    # =========================================================

    def initiate_data_preprocessing(self) -> bool:

        try:

            logger.info(
                "Starting data preprocessing..."
            )

            data = self.load_data()

            data = self.drop_columns(data)

            data = self.create_derived_features(data)

            data = self.impute_missing_values(data)

            data = self.encode_categorical_columns(data)

            data = self.scale_numeric_columns(data)

            self.save_processed_data(data)

            logger.info(
                "Data preprocessing completed "
                "successfully"
            )

            return True

        except Exception as e:

            logger.error(
                f"Error during preprocessing: {str(e)}"
            )

            raise e

In [12]:
# Production-Grade Data Preprocessing Class
import os
import pickle
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.impute import SimpleImputer

from src.customer_churn_prediction import logger


class DataPreprocessing:

    def __init__(self, config: DataPreprocessingConfig):

        self.config = config

        self.preprocessor = None

    # ======================================================
    # Load Data
    # ======================================================

    def get_raw_file_path(self) -> str:

        return os.path.join(
            self.config.raw_data_dir,
            self.config.input_file_name
        )

    def load_data(self) -> pd.DataFrame:

        raw_file_path = self.get_raw_file_path()

        if not os.path.exists(raw_file_path):

            raise FileNotFoundError(
                f"Raw data file not found: {raw_file_path}"
            )

        logger.info(
            f"Loading raw data from: {raw_file_path}"
        )

        data = pd.read_csv(raw_file_path)

        logger.info(
            f"Dataset loaded successfully: {data.shape}"
        )

        return data

    # ======================================================
    # Drop Unnecessary Columns
    # ======================================================

    def drop_columns(self,data: pd.DataFrame) -> pd.DataFrame:

        drop_cols = [
            col for col in self.config.drop_columns
            if col in data.columns
        ]

        if drop_cols:

            logger.info(
                f"Dropping columns: {drop_cols}"
            )

            data = data.drop(columns=drop_cols)

        return data

    # ======================================================
    # Create Derived Features
    # ======================================================

    def create_derived_features(self,data: pd.DataFrame) -> pd.DataFrame:

        logger.info(
            "Creating derived features..."
        )

        # Convert Total Charges
        if "Total Charges" in data.columns:

            data["Total Charges"] = pd.to_numeric(
                data["Total Charges"],
                errors="coerce"
            )

        # Average Monthly Spend
        if all(
            col in data.columns
            for col in [
                "Total Charges",
                "Tenure Months"
            ]
        ):

            data["Avg Monthly Spend"] = (
                data["Total Charges"] /
                (data["Tenure Months"] + 1)
            )

            logger.info(
                "Created feature: Avg Monthly Spend"
            )

        # Long-Term Customer
        if "Tenure Months" in data.columns:

            data["LongTermCustomer"] = (
                data["Tenure Months"] >= 24
            ).astype(int)

            logger.info(
                "Created feature: LongTermCustomer"
            )

        # High Monthly Charges
        if "Monthly Charges" in data.columns:

            threshold = data[
                "Monthly Charges"
            ].median()

            data["HighMonthlyCharges"] = (
                data["Monthly Charges"] > threshold
            ).astype(int)

            logger.info(
                "Created feature: HighMonthlyCharges"
            )

        # Total Services
        service_cols = [
            "Phone Service",
            "Online Security",
            "Online Backup",
            "Device Protection",
            "Tech Support",
            "Streaming TV",
            "Streaming Movies"
        ]

        existing_service_cols = [
            col for col in service_cols
            if col in data.columns
        ]

        if existing_service_cols:

            data["TotalServices"] = data[
                existing_service_cols
            ].apply(
                lambda row: sum(row == "Yes"),
                axis=1
            )

            logger.info(
                "Created feature: TotalServices"
            )

        logger.info(
            f"Feature engineering completed: {data.shape}"
        )

        return data

    # ======================================================
    # Build Preprocessor
    # ======================================================

    def build_preprocessor(
        self,
        numeric_columns,
        categorical_columns
    ) -> ColumnTransformer:

        logger.info(
            "Building preprocessing pipeline..."
        )

        # Numeric Pipeline
        numeric_pipeline = Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(strategy="median")
                ),
                (
                    "scaler",
                    StandardScaler()
                )
            ]
        )

        # Categorical Pipeline
        categorical_pipeline = Pipeline(
            steps=[(
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "encoder",
                    OneHotEncoder(
                        handle_unknown="ignore",sparse_output= False
                    )
                )
            ]
        )

        # Column Transformer
        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "num",
                    numeric_pipeline,
                    numeric_columns
                ),
                (
                    "cat",
                    categorical_pipeline,
                    categorical_columns
                )
            ]
        )

        logger.info(
            "Preprocessing pipeline built successfully"
        )

        return preprocessor

    # ======================================================
    # Save Preprocessor
    # ======================================================

    def save_preprocessor(self) -> None:

        preprocessor_path = os.path.join(
            self.config.root_dir,
            "preprocessor.pkl"
        )

        os.makedirs(
            self.config.root_dir,
            exist_ok=True
        )

        with open(preprocessor_path, "wb") as file:

            pickle.dump(
                self.preprocessor,
                file
            )

        logger.info(
            f"Preprocessor saved at: {preprocessor_path}"
        )

    # ======================================================
    # Save Processed Data
    # ======================================================

    def save_processed_data(
        self,
        transformed_data,
        feature_names
    ) -> None:

        processed_df = pd.DataFrame(
            transformed_data,
            columns=feature_names
        )

        os.makedirs(
            self.config.root_dir,
            exist_ok=True
        )

        processed_df.to_csv(
            self.config.processed_data_file,
            index=False
        )

        logger.info(
            f"Processed data saved to: "
            f"{self.config.processed_data_file}"
        )

        logger.info(
            f"Processed data shape: "
            f"{processed_df.shape}"
        )

    # ======================================================
    # Main Preprocessing Pipeline
    # ======================================================

    def initiate_data_preprocessing(self) -> bool:

        try:

            logger.info(
                "Starting data preprocessing..."
            )

            # Load Data
            data = self.load_data()

            # Drop Columns
            data = self.drop_columns(data)

            # Feature Engineering
            data = self.create_derived_features(data)

            # Separate Target
            target_column = self.config.target_column

            y = data[target_column]

            X = data.drop(columns=[target_column])

            # Numeric Columns
            numeric_columns = [
                col for col in self.config.numeric_columns
                if col in X.columns
            ]

            # Add Derived Numeric Features
            derived_numeric = [
                "Avg Monthly Spend",
                "TotalServices"
            ]

            numeric_columns += [
                col for col in derived_numeric
                if col in X.columns
                and col not in numeric_columns
            ]

            # Categorical Columns
            categorical_columns = [
                col for col in self.config.categorical_columns
                if col in X.columns
            ]

            # Add Derived Categorical Features
            derived_categorical = [
                "LongTermCustomer",
                "HighMonthlyCharges"
            ]

            categorical_columns += [
                col for col in derived_categorical
                if col in X.columns
                and col not in categorical_columns
            ]

            logger.info(
                f"Numeric columns: {numeric_columns}"
            )

            logger.info(
                f"Categorical columns: {categorical_columns}"
            )

            # Build Preprocessor
            self.preprocessor = self.build_preprocessor(
                numeric_columns,
                categorical_columns
            )

            # Fit + Transform
            transformed_X = self.preprocessor.fit_transform(X)

            # Get Feature Names
            feature_names = (
                self.preprocessor.get_feature_names_out()
            )


            # Final DataFrame
            processed_df = pd.DataFrame(
                transformed_X,
                columns=feature_names
            )

            processed_df[target_column] = y.values

            # Save Processed Data
            self.save_processed_data(
                processed_df,
                processed_df.columns
            )

            # Save Preprocessor
            self.save_preprocessor()

            logger.info(
                "Data preprocessing completed successfully"
            )

            return True

        except Exception as e:

            logger.error(
                f"Error during preprocessing: {str(e)}"
            )

            raise e

In [9]:
import os
import pandas as pd

from src.customer_churn_prediction import logger


class DataPreprocessing:

    def __init__(self, config: DataPreprocessingConfig):

        self.config = config

    # ======================================================
    # Raw File Path
    # ======================================================

    def get_raw_file_path(self):

        return os.path.join(
            self.config.raw_data_dir,
            self.config.input_file_name
        )

    # ======================================================
    # Load Data
    # ======================================================

    def load_data(self):

        raw_file_path = self.get_raw_file_path()

        if not os.path.exists(raw_file_path):

            raise FileNotFoundError(
                f"File not found: {raw_file_path}"
            )

        logger.info(
            f"Loading data from: {raw_file_path}"
        )

        data = pd.read_csv(raw_file_path)

        logger.info(
            f"Data shape: {data.shape}"
        )

        return data

    # ======================================================
    # Drop Columns
    # ======================================================

    def drop_columns(self, data):

        drop_cols = [
            col for col in self.config.drop_columns
            if col in data.columns
        ]

        if drop_cols:

            logger.info(
                f"Dropping columns: {drop_cols}"
            )

            data = data.drop(
                columns=drop_cols
            )

        return data

    # ======================================================
    # Feature Engineering
    # ======================================================

    def create_derived_features(self, data):

        logger.info(
            "Creating derived features..."
        )

        # Convert Total Charges
        if "Total Charges" in data.columns:

            data["Total Charges"] = pd.to_numeric(
                data["Total Charges"],
                errors="coerce"
            )

        # Avg Monthly Spend
        if all(
            col in data.columns
            for col in [
                "Total Charges",
                "Tenure Months"
            ]
        ):

            data["Avg Monthly Spend"] = (
                data["Total Charges"] /
                (data["Tenure Months"] + 1)
            )

        # Long Term Customer
        if "Tenure Months" in data.columns:

            data["LongTermCustomer"] = (
                data["Tenure Months"] >= 24
            ).astype(int)

        # High Monthly Charges
        if "Monthly Charges" in data.columns:

            threshold = data[
                "Monthly Charges"
            ].median()

            data["HighMonthlyCharges"] = (
                data["Monthly Charges"] > threshold
            ).astype(int)

        # Total Services
        service_cols = [
            "Phone Service",
            "Online Security",
            "Online Backup",
            "Device Protection",
            "Tech Support",
            "Streaming TV",
            "Streaming Movies"
        ]

        existing_cols = [
            col for col in service_cols
            if col in data.columns
        ]

        if existing_cols:

            data["TotalServices"] = data[
                existing_cols
            ].apply(
                lambda row: sum(row == "Yes"),
                axis=1
            )

        logger.info(
            f"Feature engineering completed: "
            f"{data.shape}"
        )

        return data

    # ======================================================
    # Save Cleaned Data
    # ======================================================

    def save_cleaned_data(self, data):

        os.makedirs(
            self.config.root_dir,
            exist_ok=True
        )

        output_path = (
            self.config.processed_data_file
        )

        data.to_csv(
            output_path,
            index=False
        )

        logger.info(
            f"Cleaned data saved to: "
            f"{output_path}"
        )

    # ======================================================
    # Main Pipeline
    # ======================================================

    def initiate_data_preprocessing(self):

        try:

            logger.info(
                "Starting data preprocessing..."
            )

            data = self.load_data()

            data = self.drop_columns(data)

            data = self.create_derived_features(data)

            self.save_cleaned_data(data)

            logger.info(
                "Data preprocessing completed"
            )

            return True

        except Exception as e:

            logger.error(
                f"Preprocessing error: {str(e)}"
            )

            raise e

In [10]:
try:
    config = ConfigurationManager()
    data_preprocessing_config = config.get_data_preprocessing_config()
    data_preprocessing = DataPreprocessing(config=data_preprocessing_config)
    data_preprocessing.initiate_data_preprocessing()
except Exception as e:
    raise e

[2026-05-19 19:47:21,997: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-19 19:47:22,015: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-19 19:47:22,017: INFO: common: created directory at: artifacts]
[2026-05-19 19:47:22,018: INFO: common: created directory at: artifacts/data_preprocessing]
[2026-05-19 19:47:22,019: INFO: 3780240500: Starting data preprocessing...]
[2026-05-19 19:47:22,021: INFO: 3780240500: Loading data from: artifacts/data_ingestion\data.csv]


[2026-05-19 19:47:22,105: INFO: 3780240500: Data shape: (7043, 33)]
[2026-05-19 19:47:22,110: INFO: 3780240500: Dropping columns: ['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Churn Label', 'Churn Score', 'CLTV', 'Churn Reason']]
[2026-05-19 19:47:22,122: INFO: 3780240500: Creating derived features...]
[2026-05-19 19:47:22,931: INFO: 3780240500: Feature engineering completed: (7043, 26)]
[2026-05-19 19:47:23,131: INFO: 3780240500: Cleaned data saved to: artifacts/data_preprocessing/processed_data.csv]
[2026-05-19 19:47:23,133: INFO: 3780240500: Data preprocessing completed]
